# **1. Extracción de la base de datos real desde Google Big Query**

> **Celda omitida en el repositorio.** La consulta de extracción al almacén de datos de la empresa no se publica porque contiene nombres internos de sistemas, tablas y códigos organizacionales.
>
> Su resultado es `Data/Seguro/data_real.xlsx` (no publicado): una fila por cliente, material, línea, clase de documento y mes, de enero de 2022 a septiembre de 2026, con las columnas `Fecha_Factura`, `Cod_Cliente`, `Cod_Material`, `Cod_Linea`, `cod_clase_doc_venta`, `Ventas_Usd`, `Venta_Und`, `Descuento`, `Tratamiento_Comercial`, `Detalle_Negocio`, `Provincia` y `Region`, entre otras. Los códigos de cliente y material se reemplazan por seudónimos HMAC-SHA256 en el paso siguiente.


# **2. Construcción de la base de datos de Sintética**

In [ ]:
# ---------------------------------------------------------------------------
# Definición de Variables
# ---------------------------------------------------------------------------
from pathlib import Path

RUTA = Path.cwd()
PATH = RUTA/"Data/Seguro/data_real.xlsx"
SALIDAS = RUTA/"Data/Seguro/outputs"
DIR_SEGURO = SALIDAS/"entorno_seguro"
DIR_PUBLICABLE = SALIDAS/"publicable"
DIR_FIGURAS = DIR_PUBLICABLE/"figuras"
DIR_TABLAS = DIR_PUBLICABLE/"tablas"
DIR_MODELOS = DIR_SEGURO/"modelos"
SEMILLA = 20240601

# Series externas e insumos públicos
_MACRO = RUTA/"Data/Seguro"
DIR_MACRO = _MACRO/"Macro" if (_MACRO/"Macro").is_dir() else _MACRO   # admite la carpeta anidada del .rar
RUTA_COMPLETADOS = RUTA/"Data/Seguro/exogenas_completadas.csv"
DIR_PUBLICO = RUTA/"Data/Publico"

# ---------------------------------------------------------------------------
# Período y particiones
# ---------------------------------------------------------------------------
HOJA_EXCEL = 0                    # primera hoja del Excel
PERIODO_INICIO = "2022-01"        # primer mes de la extracción
PERIODO_FIN = "2026-09"           # último mes completo
FIN_ENTRENAMIENTO = "2025-09"     # tipos de relación (ADI y CV²) solo con datos hasta aquí
FIN_VALIDACION = "2026-03"        # la prueba es 2026-04 a 2026-09

# ---------------------------------------------------------------------------
# Columnas del Excel -> nombres internos (se aceptan varios alias)
# ---------------------------------------------------------------------------
ALIAS_COLUMNAS = {
    "fecha":       ["Fecha_Factura", "Fecha", "Fecha_Documento", "Fecha_Pedido"],
    "cliente":     ["Cod_Cliente", "Cliente"],
    "sku":         ["Cod_Linea"],                       # unidad de producto del pronóstico: LÍNEA
    "material":    ["Cod_Material"],
    "clase_doc":   ["cod_clase_doc_venta", "Clase_Documento", "Clase_Doc"],
    "ventas_usd":  ["Ventas_Usd", "Ventas_USD"],
    "unidades":    ["Venta_Und", "Unidades", "Cantidad"],
    "descuento":   ["Descuento"],
    "tratamiento": ["Tratamiento_Comercial"],
    "detalle_neg": ["Detalle_Negocio"],
    "provincia":   ["Provincia"],
    "region":      ["Region"],
}
COLUMNAS_OBLIGATORIAS = ["fecha", "cliente", "sku", "clase_doc", "ventas_usd"]
ATRIBUTOS_CLIENTE = ["tratamiento", "detalle_neg", "provincia", "region"]
COLUMNAS_A_SEUDONIMIZAR = ["cliente", "material"]      # se reemplazan por HMAC-SHA256

# ---------------------------------------------------------------------------
# Generación sintética
# ---------------------------------------------------------------------------
CTGAN_EPOCHS = 300
CTGAN_BATCH = 500
SINTETICO_MAX_ITER_MEJORA = 2     # 300 -> 600 -> 1 200 épocas como máximo
RUIDO_ESCALA_PAR = 0.15           # desviación (en log) añadida a la escala de cada relación

# Criterios de aceptación DEFINIDOS ANTES de generar
CRITERIOS_ACEPTACION = {
    "ks_max_continuas": 0.10,
    "tvd_max_categoricas": 0.10,
    "dif_tasa_actividad_pp": 2.0,
    "dif_pct_ceros_pp": 2.0,
    "wape_ventas_mensuales_agregadas": 0.15,
    "dif_media_acf_lag1": 0.10,
    "dif_media_acf_lag12": 0.10,
    "dif_max_correlacion": 0.15,
    "max_prop_dcr_bajo_p5": 0.10,
}

for nombre, ruta in [("Base real", PATH), ("Datos_Macro", DIR_MACRO), ("exogenas_completadas", RUTA_COMPLETADOS)]:
    print(f"{nombre:22s} {'OK' if ruta.exists() else 'NO ENCONTRADO'} -> {ruta}")

In [ ]:
import hashlib
import hmac
import json
import pickle
import secrets
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from scipy.stats import ks_2samp
from sklearn.neighbors import NearestNeighbors
from statsmodels.tsa.stattools import acf
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)

In [ ]:
for d in [DIR_SEGURO, DIR_PUBLICABLE, DIR_FIGURAS, DIR_TABLAS, DIR_MODELOS]:
    d.mkdir(parents=True, exist_ok=True)


class Bitacora:
    """Registro secuencial de cada transformación (evidencia para la sección 4.2)."""

    def __init__(self, nombre):
        self.nombre = nombre
        self.pasos = []

    def registrar(self, paso, descripcion, filas_antes=None, filas_despues=None, **extra):
        self.pasos.append({
            "n": len(self.pasos) + 1, "paso": paso, "descripcion": descripcion,
            "filas_antes": filas_antes, "filas_despues": filas_despues, **extra,
        })
        print(f"  [{self.nombre} {len(self.pasos):02d}] {paso}: {descripcion}"
              + (f" ({filas_antes} -> {filas_despues})" if filas_antes is not None else ""))

    def guardar(self, ruta):
        pd.DataFrame(self.pasos).to_csv(ruta, index=False, encoding="utf-8-sig")


def guardar_json(obj, ruta):
    def _conv(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, (pd.Timestamp, pd.Period)):
            return str(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        return str(o)
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=_conv)


PALETA = ["#1F6FB4", "#D1572B", "#1A9A77", "#8E5BC4", "#B8860B"]
GRISES = ["#52514e", "#8a8984", "#b5b4ae"]
COLOR_MODELO = {"Random Forest": PALETA[0], "MLP": PALETA[1], "XGBoost": PALETA[2],
                "LightGBM": PALETA[3], "CatBoost": PALETA[4]}
COLOR_REAL, COLOR_SINT = "#0b0b0b", PALETA[0]          # real/observado: tinta; sintética: serie 1
COLOR_OBS = "#0b0b0b"                                      # valor observado en toda la tesis
COLOR_POS, COLOR_NEG = PALETA[0], PALETA[1]                # efecto o venta positiva / negativa
COLOR_PARTICION = {"entrenamiento": "#EAF1F8", "validacion": "#FBF3DF", "prueba": "#FBE9E2"}
ESTILO_REF = {"Método empresa (línea = prom. 6 m)": (GRISES[0], "--"), "Croston–SBA": (GRISES[1], "-."),
              "Método empresa (línea real conocida)": (GRISES[0], ":"), "Media móvil 6": (GRISES[2], "--")}
NOMBRE_CORTO = {"Método empresa (línea = prom. 6 m)": "Método empresa",
                "Método empresa (línea real conocida)": "Empresa (línea conocida)"}


def estilo():
    """Estilo único de todas las figuras de la tesis: Liberation Sans (métrica de Arial), sin títulos
    (la descripción va en el pie de figura), cuadrícula tenue y ejes recesivos."""
    plt.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 300, "font.family": ["Arial", "Liberation Sans", "DejaVu Sans"],
        "font.size": 14, "axes.titlesize": 14, "axes.labelsize": 14, "xtick.labelsize": 12.5, "ytick.labelsize": 12.5,
        "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRISES[1],
        "axes.labelcolor": "#0b0b0b", "xtick.color": GRISES[0], "ytick.color": GRISES[0],
        "axes.grid": True, "grid.alpha": 0.25, "grid.color": GRISES[2], "legend.frameon": False,
        "legend.fontsize": 12.5, "lines.linewidth": 2, "axes.prop_cycle": plt.cycler(color=PALETA),
    })


def guardar_figura(fig, nombre):
    destino = DIR_FIGURAS / f"{nombre}.png"
    fig.tight_layout()
    fig.savefig(destino, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(destino), width=700))
    return destino


print("Carpetas listas en", SALIDAS)

## Paso 1. Auditoría, seudonimización y panel cliente–línea–mes (Fase A)

1. Lee la extracción de GCP y guarda una copia de auditoría (la primera lectura del Excel tarda unos minutos; después usa `extraccion_cruda.pkl`).
2. Sustituye los códigos de cliente y material por seudónimos **HMAC-SHA256** (clave en `DIR_SEGURO/clave_hmac.key`; si no existe, se crea).
3. Conserva las **17 clases de documento**: la variable es la **venta neta**.
4. Suma la venta neta por cliente, línea y mes e inserta los meses sin movimiento.
5. Clasifica las relaciones por ADI y CV² y marca los atípicos sin eliminarlos.

In [ ]:
def _mapear_columnas(df):
    mapa, faltantes = {}, []
    for canon, alias in ALIAS_COLUMNAS.items():
        hallada = next((a for a in alias if a in df.columns), None)
        if hallada is not None:
            mapa[hallada] = canon
        elif canon in COLUMNAS_OBLIGATORIAS:
            faltantes.append(canon)
    if faltantes:
        raise KeyError(f"No se encontraron columnas para {faltantes}. Columnas: {list(df.columns)}")
    return mapa


def seudonimizar(df, bit):
    """HMAC-SHA256 con clave aleatoria de 256 bits guardada solo en el entorno seguro."""
    ruta = DIR_SEGURO / "clave_hmac.key"
    if not ruta.exists():
        ruta.write_bytes(secrets.token_bytes(32))
    clave = ruta.read_bytes()
    for c in COLUMNAS_A_SEUDONIMIZAR:
        mapa = {u: hmac.new(clave, f"{c}:{u}".encode(), hashlib.sha256).hexdigest()[:16] for u in df[c].unique()}
        df[c] = df[c].map(mapa)
    bit.registrar("seudonimizacion", "Cod_Cliente y Cod_Material llegaban en claro; se reemplazan por HMAC-SHA256",
                len(df), len(df))
    return df


def auditar_y_preprocesar():
    bit = Bitacora("A")
    rep = {}

    # 1. Carga de la extracción GCP (enero 2022 – septiembre 2026) y copia de auditoría ----------
    cache = DIR_SEGURO / "extraccion_cruda.pkl"   # lectura rápida de la misma extracción
    if cache.exists() and cache.stat().st_mtime > PATH.stat().st_mtime:
        crudo = pd.read_pickle(cache)
    else:
        crudo = pd.read_excel(PATH, sheet_name=HOJA_EXCEL)
        crudo.to_pickle(cache)
    crudo = crudo[pd.to_datetime(crudo["Fecha_Factura"]) >= pd.Period(PERIODO_INICIO, "M").start_time]
    crudo.to_pickle(DIR_SEGURO / "copia_auditoria_original.pkl")
    rep["columnas_originales"] = list(crudo.columns)
    rep["tipos_originales"] = crudo.dtypes.astype(str).to_dict()
    rep["nulos_originales"] = crudo.isna().sum().to_dict()
    bit.registrar("carga", f"Lectura de la extracción GCP ({PERIODO_INICIO} a {PERIODO_FIN}) y copia de auditoría", None, len(crudo))

    mapa = _mapear_columnas(crudo)
    df = crudo.rename(columns=mapa)[list(mapa.values())].copy()

    # 2. Tipos, identificadores y seudonimización --------------------------------------------------
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
    for c in ["ventas_usd", "unidades", "descuento"]:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in ["cliente", "sku", "material", "clase_doc"] + [a for a in ATRIBUTOS_CLIENTE if a in df]:
        df[c] = df[c].astype(str).str.strip()
    df = seudonimizar(df, bit)
    n0 = len(df)
    df = df.dropna(subset=["fecha", "ventas_usd"])
    bit.registrar("tipos_nulos", "Conversión de tipos; se descartan filas sin fecha o sin monto", n0, len(df))
    df = df[df["fecha"] <= pd.Period(PERIODO_FIN, "M").end_time]

    # 3. Auditoría de clases documentales (todas se conservan: la variable es venta neta) ----------
    pc = df.groupby("clase_doc").agg(registros=("ventas_usd", "size"),
                                     pct_registros_negativos=("ventas_usd", lambda s: 100 * (s < 0).mean()))
    pc["pct_registros"] = 100 * pc["registros"] / pc["registros"].sum()
    pc.sort_values("registros", ascending=False).round(2).to_csv(
        DIR_TABLAS / "A_auditoria_clase_documental.csv", encoding="utf-8-sig")
    rep["clases_documentales"] = int(df["clase_doc"].nunique())
    rep["registros"] = len(df)
    rep["pct_registros_negativos"] = round(100 * (df["ventas_usd"] < 0).mean(), 2)
    n_dup = int(df.duplicated().sum())
    rep["duplicados_exactos"] = n_dup
    bit.registrar("clases", f"Se conservan las {rep['clases_documentales']} clases documentales (pedidos, devoluciones, "
                f"notas de crédito, refacturaciones); {rep['pct_registros_negativos']} % de registros negativos; "
                f"{n_dup} duplicados exactos (se conservan: sin número de documento son legítimos)", len(df), len(df))
    if "descuento" in df:
        rep["pct_lineas_con_descuento"] = round(100 * (df["descuento"].fillna(0) != 0).mean(), 2)
    df.to_pickle(DIR_SEGURO / "lineas_extraccion.pkl")

    # 4. Agregación cliente–línea–mes (venta neta y unidades netas) -------------------------------
    df["mes"] = df["fecha"].dt.to_period("M")
    mensual = df.groupby(["cliente", "sku", "mes"]).agg(
        ventas_usd=("ventas_usd", "sum"), unidades=("unidades", "sum"), n_lineas=("ventas_usd", "size"),
        n_clases=("clase_doc", "nunique")).reset_index()
    mensual["ventas_usd"] = mensual["ventas_usd"].round(2)
    bit.registrar("agregacion", "Suma de la venta neta y las unidades netas por cliente–línea–mes", len(df), len(mensual))

    # 5. Panel completo desde el primer mes con movimiento neto distinto de cero --------------------
    meses = pd.period_range(PERIODO_INICIO, PERIODO_FIN, freq="M")
    con_mov = mensual[mensual["ventas_usd"] != 0]
    primera = con_mov.groupby(["cliente", "sku"])["mes"].min().rename("primer_mes").reset_index()
    filas = [pd.DataFrame({"cliente": r.cliente, "sku": r.sku, "mes": meses[meses >= r.primer_mes]})
            for r in primera.itertuples(index=False)]
    panel = pd.concat(filas, ignore_index=True).merge(mensual, on=["cliente", "sku", "mes"], how="left")
    for c in ["ventas_usd", "unidades", "n_lineas", "n_clases"]:
        panel[c] = panel[c].fillna(0)
    panel["activo"] = (panel["ventas_usd"] != 0).astype(int)
    panel = panel.merge(primera, on=["cliente", "sku"])
    moda = df.groupby("cliente")[[a for a in ATRIBUTOS_CLIENTE if a in df]].agg(lambda x: x.mode().iat[0])
    panel = panel.merge(moda, left_on="cliente", right_index=True, how="left")
    panel["par_id"] = panel.groupby(["cliente", "sku"]).ngroup()
    panel["linea"] = panel["sku"]
    bit.registrar("panel", "Inserción de meses sin movimiento desde el primer mes con venta neta distinta de cero",
                len(mensual), len(panel))

    act = panel[panel["activo"] == 1]
    rep["cifras"] = {
        "registros": rep["registros"], "clientes": df["cliente"].nunique(), "lineas": df["sku"].nunique(),
        "materiales": df["material"].nunique(), "relaciones_cliente_linea": int(primera.shape[0]),
        "observaciones_relacion_mes": len(panel),
        "pct_meses_sin_movimiento": round(100 * (1 - panel["activo"].mean()), 2),
        "pct_meses_con_venta_neta_negativa": round(100 * (panel["ventas_usd"] < 0).mean(), 2),
        "pct_meses_activos_negativos": round(100 * (act["ventas_usd"] < 0).mean(), 2),
        "negativos_sobre_positivos_pct": round(100 * -act.loc[act.ventas_usd < 0, "ventas_usd"].sum()
                                            / act.loc[act.ventas_usd > 0, "ventas_usd"].sum(), 2),
    }
    print(pd.Series(rep["cifras"]))

    # 6. Intermitencia (Syntetos–Boylan) con meses de entrenamiento --------------------------------
    panel = clasificar_intermitencia(panel)

    # 7. Atípicos: marcado (no eliminación) por IQR en log|y| por línea y percentil 99 ----------------
    pos = panel["activo"] == 1
    lv = np.log(panel.loc[pos, "ventas_usd"].abs())
    q = panel.loc[pos].assign(lv=lv).groupby("sku")["lv"].quantile([0.25, 0.75]).unstack()
    q.columns = ["q1", "q3"]
    tmp = panel.loc[pos, ["sku"]].join(q, on="sku")
    iqr = tmp["q3"] - tmp["q1"]
    panel["atipico_iqr"] = 0
    panel.loc[pos, "atipico_iqr"] = ((lv > tmp["q3"] + 1.5 * iqr) | (lv < tmp["q1"] - 1.5 * iqr)).astype(int)
    p99 = panel.loc[pos].groupby("sku")["ventas_usd"].transform(lambda s: s.abs().quantile(0.99))
    panel["atipico_p99"] = 0
    panel.loc[pos, "atipico_p99"] = (panel.loc[pos, "ventas_usd"].abs() > p99).astype(int)
    rep["atipicos"] = {"pct_iqr": round(100 * panel.loc[pos, "atipico_iqr"].mean(), 2),
                       "pct_p99": round(100 * panel.loc[pos, "atipico_p99"].mean(), 2)}
    bit.registrar("atipicos", f"Marcado IQR (log|y|, por línea): {rep['atipicos']['pct_iqr']} % de meses con movimiento; "
                f"P99: {rep['atipicos']['pct_p99']} %. Se conservan", len(panel), len(panel))

    panel = panel.sort_values(["par_id", "mes"]).reset_index(drop=True)
    panel.to_pickle(DIR_SEGURO / "panel_real.pkl")
    bit.guardar(DIR_TABLAS / "A_bitacora_transformaciones.csv")
    guardar_json(rep, DIR_PUBLICABLE / "A_reporte_auditoria.json")
    return panel, rep


def clasificar_intermitencia(panel, fin=None):
    """ADI y CV² del tamaño |y| de los meses con movimiento (Syntetos et al., 2005), con datos <= fin."""
    fin = pd.Period(fin or FIN_ENTRENAMIENTO, "M")
    h = panel[panel["mes"] <= fin]
    g = h.groupby("par_id")
    n_meses, n_pos = g.size(), g["activo"].sum()
    tam = h[h["activo"] == 1].assign(a=lambda d: d["ventas_usd"].abs()).groupby("par_id")["a"]
    est = pd.DataFrame({"n_meses_ent": n_meses, "n_pos_ent": n_pos, "cv2": (tam.std(ddof=0) / tam.mean()) ** 2})
    est["adi"] = est["n_meses_ent"] / est["n_pos_ent"].replace(0, np.nan)

    def _cls(r):
        if pd.isna(r.adi) or r.n_pos_ent < 2:
            return "nueva/insuficiente"
        if r.adi < 1.32:
            return "suave" if r.cv2 < 0.49 else "errática"
        return "intermitente" if r.cv2 < 0.49 else "irregular (lumpy)"
    est["segmento_sb"] = est.apply(_cls, axis=1)
    freq = est["n_pos_ent"] / est["n_meses_ent"]
    est["segmento_freq"] = pd.cut(freq, [-0.01, 0.10, 0.30, 0.60, 1.0],
                                labels=["muy baja (<=10%)", "baja (10–30%)", "media (30–60%)", "alta (>60%)"])
    est["segmento_freq"] = est["segmento_freq"].cat.add_categories("nueva").fillna("nueva")
    panel = panel.drop(columns=[c for c in ["segmento_sb", "segmento_freq", "adi", "cv2"] if c in panel])
    panel = panel.merge(est[["segmento_sb", "segmento_freq", "adi", "cv2"]], left_on="par_id",
                        right_index=True, how="left")
    panel["segmento_sb"] = panel["segmento_sb"].fillna("nueva/insuficiente")
    panel["segmento_freq"] = panel["segmento_freq"].astype(object).fillna("nueva")
    return panel


In [ ]:
panel_real, reporte = auditar_y_preprocesar()
print(panel_real.shape)
panel_real.groupby("par_id").segmento_sb.first().value_counts()

## Paso 2. Generador sintético híbrido (Fase B)

- **Esqueleto:** conserva las relaciones cliente–línea y su mes de inicio, con códigos nuevos (C0001, L01).
- **Actividad:** cadena de Markov con contracción hacia el tipo de relación y un factor común mensual.
- **Monto y signo:** CTGAN aprende el residuo del monto, el signo y el precio implícito; un AR(1) latente impone la autocorrelación.

In [ ]:
def _logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def _sig(x):
    return 1 / (1 + np.exp(-x))


class GeneradorHibrido:
    def __init__(self, epochs=CTGAN_EPOCHS, semilla=SEMILLA, k_prior=4.0, verbose=False):
        self.epochs, self.semilla, self.k_prior, self.verbose = epochs, semilla, k_prior, verbose
        self.rng = np.random.default_rng(semilla)

    def ajustar(self, panel, fin):
        self.fin = pd.Period(fin, "M")
        d = panel[panel["mes"] <= self.fin].sort_values(["par_id", "mes"]).copy()
        d["prev"] = d.groupby("par_id")["activo"].shift(1)
        t = d.dropna(subset=["prev"])

        # (2) actividad: transición por relación con contracción hacia el segmento + factor mensual
        base = t.groupby(["segmento_sb", "prev"])["activo"].mean().rename("p_seg")
        cnt = t.groupby(["par_id", "segmento_sb", "prev"])["activo"].agg(["sum", "count"]).reset_index()
        cnt = cnt.join(base, on=["segmento_sb", "prev"])
        cnt["p"] = (cnt["sum"] + self.k_prior * cnt["p_seg"]) / (cnt["count"] + self.k_prior)
        self.p_trans = cnt.set_index(["par_id", "prev"])["p"]
        self.p_seg = base
        # factor común mensual de actividad: desvío logit de cada mes frente a lo esperado por transición
        t = t.join(self.p_trans.rename("p_i"), on=["par_id", "prev"])
        obs = t.groupby("mes")["activo"].mean()
        esp = t.groupby("mes")["p_i"].mean()
        self.f_act = (_logit(obs) - _logit(esp)).to_dict()

        # (3) montos: escala por relación (contraída hacia la línea) y factor común mensual
        a = d[d["activo"] == 1].copy()
        a["la"] = np.log(a["ventas_usd"].abs())
        mu = a.groupby("par_id")["la"].agg(["mean", "count"])
        mu_lin = a.groupby("sku")["la"].mean()
        sku_par = d.groupby("par_id")["sku"].first()
        w = mu["count"] / (mu["count"] + 3)
        self.mu_par = w * mu["mean"] + (1 - w) * sku_par.loc[mu.index].map(mu_lin).values
        a["r"] = a["la"] - a["par_id"].map(self.mu_par)
        # factor común mensual del monto, ponderado por la escala de la relación (las grandes mueven el total)
        a["w"] = np.exp(a["par_id"].map(self.mu_par))
        self.f_monto = (a.groupby("mes").apply(lambda g: np.average(g["r"], weights=g["w"]))).to_dict()
        a["e"] = a["r"] - a["mes"].map(self.f_monto)
        self.cortes = np.quantile(self.mu_par.values, [0.2, 0.4, 0.6, 0.8])
        self.escala_q = pd.Series(np.digitize(self.mu_par.values, self.cortes).astype(str), index=self.mu_par.index)
        a["escala_q"] = a["par_id"].map(self.escala_q)
        a["mes_cal"] = a["mes"].dt.month.astype(str)
        a["prev"] = a["prev"].fillna(-1).astype(int).astype(str)
        a["signo"] = np.where(a["ventas_usd"] < 0, "neg", "pos")
        u = a["unidades"].abs()
        a["lp"] = np.where(u > 0, np.log(a["ventas_usd"].abs() / u.replace(0, np.nan)), np.nan)
        self.mu_precio = a.groupby("sku")["lp"].mean()
        a["r_p"] = (a["lp"] - a["sku"].map(self.mu_precio)).fillna(0)
        self.tabla = a[["escala_q", "segmento_sb", "mes_cal", "prev", "signo", "e", "r_p"]].reset_index(drop=True)
        # momentos reales del residuo por quintil de escala (para calibrar la salida de CTGAN)
        self.mom = self.tabla.groupby("escala_q")["e"].agg(["mean", "std"])

        a = a.sort_values(["par_id", "mes"])
        a["e_ant"] = a.groupby("par_id")["e"].shift(1)
        self.rho = float(np.clip(a[["e", "e_ant"]].dropna().corr().iloc[0, 1], 0, 0.95))

        import torch
        from ctgan import CTGAN
        torch.manual_seed(self.semilla)
        self.ctgan = CTGAN(epochs=self.epochs, batch_size=CTGAN_BATCH, verbose=self.verbose, pac=10,
                           enable_gpu=False)
        self.ctgan.fit(self.tabla, discrete_columns=["escala_q", "segmento_sb", "mes_cal", "prev", "signo"])
        self.esqueleto = d.groupby("par_id").agg(cliente=("cliente", "first"), sku=("sku", "first"),
                                                 primer_mes=("primer_mes", "first"),
                                                 segmento_sb=("segmento_sb", "first"),
                                                 segmento_freq=("segmento_freq", "first"))
        return self

    def generar(self):
        rng = self.rng
        meses = pd.period_range(PERIODO_INICIO, self.fin, freq="M")
        esq = self.esqueleto
        cl, sk = esq["cliente"].unique(), esq["sku"].unique()
        mapa_c = dict(zip(rng.permutation(cl), [f"C{i:04d}" for i in range(1, len(cl) + 1)]))
        mapa_s = dict(zip(rng.permutation(sk), [f"L{i:02d}" for i in range(1, len(sk) + 1)]))
        filas = []
        for par, r in esq.iterrows():
            rm = meses[meses >= r.primer_mes]
            act = np.zeros(len(rm), int)
            act[0] = 1
            for i in range(1, len(rm)):
                p = self.p_trans.get((par, act[i - 1]), self.p_seg.get((r.segmento_sb, act[i - 1]), 0.3))
                act[i] = rng.random() < _sig(_logit(p) + self.f_act.get(rm[i], 0.0))
            filas.append(pd.DataFrame({"par_id": par, "mes": rm, "activo": act}))
        s = pd.concat(filas, ignore_index=True).join(esq, on="par_id")
        s["prev"] = s.groupby("par_id")["activo"].shift(1).fillna(-1).astype(int).astype(str)
        s["mes_cal"] = s["mes"].dt.month.astype(str)
        s["escala_q"] = s["par_id"].map(self.escala_q).fillna("2")

        idx = s.index[s["activo"] == 1]
        pool = self.ctgan.sample(max(4 * len(idx), 50_000)).reset_index(drop=True)
        # calibración de momentos: media y desviación del residuo iguales a las reales en cada quintil
        for q, g in pool.groupby("escala_q"):
            if q in self.mom.index and g["e"].std() > 0:
                pool.loc[g.index, "e"] = (g["e"] - g["e"].mean()) / g["e"].std() * self.mom.loc[q, "std"] \
                    + self.mom.loc[q, "mean"]
        e = np.full(len(s), np.nan)
        rp = np.zeros(len(s))
        sg = np.array(["pos"] * len(s), dtype=object)
        pend = s.loc[idx]
        for claves in (["escala_q", "segmento_sb", "mes_cal"], ["escala_q", "segmento_sb"], ["escala_q"], []):
            if pend.empty:
                break
            kp = pool[claves].astype(str).agg("|".join, axis=1) if claves else pd.Series("*", pool.index)
            ks = pend[claves].astype(str).agg("|".join, axis=1) if claves else pd.Series("*", pend.index)
            gp = kp.groupby(kp).indices
            hechos = []
            for k, g in pend.groupby(ks):
                ip = gp.get(k)
                if ip is None or (len(ip) < 5 and claves):
                    continue
                el = rng.choice(ip, size=len(g), replace=True)
                e[g.index] = pool.loc[el, "e"].values
                rp[g.index] = pool.loc[el, "r_p"].values
                sg[g.index] = pool.loc[el, "signo"].values
                hechos.append(g.index)
            if hechos:
                pend = pend.drop(np.concatenate(hechos))
        s["e"] = e
        for par, g in s.loc[idx].groupby("par_id"):
            if len(g) < 3:
                continue
            z = np.zeros(len(g))
            z[0] = rng.normal()
            for i in range(1, len(g)):
                z[i] = self.rho * z[i - 1] + np.sqrt(1 - self.rho ** 2) * rng.normal()
            s.loc[g.index, "e"] = np.sort(g["e"].values)[np.argsort(np.argsort(z))]

        ruido = pd.Series(rng.normal(0, RUIDO_ESCALA_PAR, len(esq)), index=esq.index)
        escala = s["par_id"].map(self.mu_par).fillna(self.mu_par.median()) + s["par_id"].map(ruido)
        log_abs = escala + s["mes"].map(self.f_monto).fillna(0) + s["e"].fillna(0)
        signo = np.where(sg == "neg", -1.0, 1.0)
        s["ventas_usd"] = np.where(s["activo"] == 1, signo * np.exp(log_abs), 0.0).round(2)
        precio = np.exp(s["sku"].map(self.mu_precio).fillna(self.mu_precio.median()) + rp)
        s["unidades"] = np.where(s["activo"] == 1, signo * np.maximum(1, np.round(np.abs(s["ventas_usd"]) / precio)), 0)
        s["cliente"] = s["cliente"].map(mapa_c)
        s["sku"] = s["sku"].map(mapa_s)
        s["n_lineas"] = s["activo"]
        return s[["par_id", "cliente", "sku", "mes", "ventas_usd", "unidades", "n_lineas", "activo", "primer_mes",
                  "segmento_sb", "segmento_freq"]].sort_values(["par_id", "mes"]).reset_index(drop=True)


### Validación de fidelidad y privacidad (Fase C)
KS, TVD, autocorrelación, totales mensuales, correlaciones y distancia al registro real más cercano (DCR), frente a `CRITERIOS_ACEPTACION`.

In [ ]:
from scipy.stats import ks_2samp
from sklearn.neighbors import NearestNeighbors
from statsmodels.tsa.stattools import acf



def _tvd(a, b):
    pa = a.value_counts(normalize=True)
    pb = b.value_counts(normalize=True)
    idx = pa.index.union(pb.index)
    return float(0.5 * np.abs(pa.reindex(idx, fill_value=0) - pb.reindex(idx, fill_value=0)).sum())


def _ks(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    return float(ks_2samp(a, b).statistic)


def _por_par(p):
    g = p.groupby("par_id")
    act = p[p["activo"] == 1].sort_values(["par_id", "mes"])
    gap = act.groupby("par_id")["mes"].diff().apply(lambda x: x.n if pd.notna(x) else np.nan)
    ult = act.groupby("par_id")["mes"].max()
    fin = p["mes"].max()
    return {
        "tasa_actividad_par": g["activo"].mean(),
        "compras_por_par": g["activo"].sum(),
        "ventas_totales_par_log": np.log1p(g["ventas_usd"].sum().clip(lower=0)),
        "intervalo_entre_compras": gap.dropna(),
        "recencia_final": (fin - ult).apply(lambda x: x.n),
        "pares_por_cliente": p.groupby("cliente")["sku"].nunique(),
    }


def _rasgos_fila(p):
    p = p.sort_values(["par_id", "mes"]).copy()
    p["lv"] = np.sign(p["ventas_usd"]) * np.log1p(p["ventas_usd"].abs())
    g = p.groupby("par_id")["lv"]
    p["lv_lag1"] = g.shift(1)
    p["lv_media3"] = g.transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    p["mes_cal"] = p["mes"].dt.month
    cols = ["lv", "lv_lag1", "lv_media3", "mes_cal"]
    if "unidades" in p:
        p["lu"] = np.sign(p["unidades"]) * np.log1p(p["unidades"].abs())
        cols.insert(1, "lu")
    return p, cols


def _acf_media(p, lag):
    vals = []
    for _, s in p.groupby("par_id")["ventas_usd"]:
        if len(s) >= max(24, 2 * lag) and s.std() > 0:
            vals.append(acf(s.values, nlags=lag, fft=True)[lag])
    return float(np.nanmean(vals)) if vals else np.nan, len(vals)


def validar(real, sint, semilla=SEMILLA):
    fin = sint["mes"].max()
    real = real[real["mes"] <= fin].copy()
    cr = CRITERIOS_ACEPTACION
    filas = []

    def add(dim, metrica, valor, umbral=None, cumple=None, nota=""):
        if cumple is None and umbral is not None and pd.notna(valor):
            cumple = bool(valor <= umbral)
        filas.append({"dimension": dim, "metrica": metrica, "valor": valor, "umbral": umbral,
                      "cumple": cumple, "nota": nota})

    # ---- Fidelidad marginal
    ra, sa = real[real["activo"] == 1], sint[sint["activo"] == 1]
    add("Fidelidad", "KS log|venta neta| (meses con movimiento)",
        _ks(np.log(ra["ventas_usd"].abs()), np.log(sa["ventas_usd"].abs())), cr["ks_max_continuas"],
        nota="CTGAN + factor común")
    add("Fidelidad", "TVD signo de la venta neta (positiva/negativa)",
        _tvd(np.sign(ra["ventas_usd"]), np.sign(sa["ventas_usd"])), cr["tvd_max_categoricas"], nota="CTGAN")
    ru, su = ra[ra["unidades"].abs() > 0], sa[sa["unidades"].abs() > 0]
    add("Fidelidad", "KS log|unidades netas|", _ks(np.log(ru["unidades"].abs()), np.log(su["unidades"].abs())),
        cr["ks_max_continuas"], nota="CTGAN")
    add("Fidelidad", "KS log(precio implícito)",
        _ks(np.log((ru["ventas_usd"] / ru["unidades"]).abs()), np.log((su["ventas_usd"] / su["unidades"]).abs())),
        cr["ks_max_continuas"], nota="CTGAN")
    pr, ps = _por_par(real), _por_par(sint)
    for k in ["tasa_actividad_par", "compras_por_par", "ventas_totales_par_log",
              "intervalo_entre_compras", "recencia_final"]:
        add("Frecuencia/recencia" if k in ("compras_por_par", "intervalo_entre_compras", "recencia_final")
            else "Fidelidad", f"KS {k}", _ks(pr[k], ps[k]), cr["ks_max_continuas"],
            nota="Componente Markov" if k != "ventas_totales_par_log" else "Markov + CTGAN")
    add("Relaciones cliente–línea", "KS pares por cliente", _ks(pr["pares_por_cliente"], ps["pares_por_cliente"]),
        cr["ks_max_continuas"], nota="Preservado por diseño (esqueleto)")
    add("Fidelidad", "TVD mes calendario de meses activos",
        _tvd(ra["mes"].dt.month, sa["mes"].dt.month), cr["tvd_max_categoricas"])
    add("Fidelidad", "TVD segmento de frecuencia (filas activas)",
        _tvd(ra["segmento_freq"].astype(str), sa["segmento_freq"].astype(str)), cr["tvd_max_categoricas"])

    # ---- Ceros / actividad
    d_act = abs(real["activo"].mean() - sint["activo"].mean()) * 100
    add("Ceros y actividad", "|Δ tasa de actividad| (pp)", d_act, cr["dif_tasa_actividad_pp"])
    add("Ceros y actividad", "|Δ % meses sin pedido| (pp)", d_act, cr["dif_pct_ceros_pp"],
        nota="Complemento exacto de la tasa de actividad")

    # ---- Ventas agregadas mensuales
    mr = real.groupby("mes")["ventas_usd"].sum()
    ms = sint.groupby("mes")["ventas_usd"].sum().reindex(mr.index, fill_value=0)
    wape_m = float(np.abs(mr - ms).sum() / mr.sum())
    add("Estructura temporal", "WAPE ventas mensuales agregadas", wape_m, cr["wape_ventas_mensuales_agregadas"])
    add("Estructura temporal", "Correlación ventas mensuales agregadas", float(np.corrcoef(mr, ms)[0, 1]),
        nota="Informativa")
    ar = real.groupby("mes")["activo"].mean()
    asn = sint.groupby("mes")["activo"].mean().reindex(ar.index)
    add("Estructura temporal", "Máx |Δ tasa de actividad mensual| (pp)", float((ar - asn).abs().max() * 100),
        nota="Informativa")
    for lag, key in [(1, "dif_media_acf_lag1"), (12, "dif_media_acf_lag12")]:
        a_r, n_r = _acf_media(real, lag)
        a_s, n_s = _acf_media(sint, lag)
        add("Estructura temporal", f"|Δ ACF media lag {lag}| (real {a_r:.3f}, sint {a_s:.3f}; n={n_r})",
            abs(a_r - a_s), cr[key])
    perf_r = ra.groupby(ra["mes"].dt.month)["ventas_usd"].sum()
    perf_s = sa.groupby(sa["mes"].dt.month)["ventas_usd"].sum()
    add("Estructura temporal", "TVD perfil estacional de ventas",
        float(0.5 * np.abs(perf_r / perf_r.sum() - (perf_s / perf_s.sum()).reindex(perf_r.index, fill_value=0)).sum()),
        cr["tvd_max_categoricas"])

    # ---- Correlaciones (Spearman) entre rasgos fila
    rr, cols = _rasgos_fila(real)
    ss, _ = _rasgos_fila(sint)
    cr_r = rr[cols].corr("spearman")
    cr_s = ss[cols].corr("spearman")
    dmax = float((cr_r - cr_s).abs().values.max())
    add("Relaciones", "Máx |Δρ Spearman| entre rasgos", dmax, cr["dif_max_correlacion"])
    pd.concat({"real": cr_r, "sintetico": cr_s, "diferencia": cr_r - cr_s}).to_csv(
        DIR_TABLAS / "C_correlaciones_real_vs_sintetico.csv", encoding="utf-8-sig")

    # ---- Variabilidad y persistencia dentro de cada relación (informativos)
    def _cv2(p):
        z = p[p["activo"] == 1].assign(z=lambda d: d["ventas_usd"].abs()).groupby("par_id")["z"]
        return float((z.std(ddof=0) / z.mean()).pow(2).median())
    add("Estructura temporal", f"Mediana CV² del monto por relación (real {_cv2(real):.3f})", _cv2(sint),
        nota="Informativa: variabilidad mes a mes dentro de la relación")

    # ---- Privacidad: DCR (distancia al registro real más cercano)
    rng = np.random.default_rng(semilla)
    pares = real["par_id"].unique()
    hold = set(rng.choice(pares, size=len(pares) // 2, replace=False))
    act_r = rr[(rr["activo"] == 1)].dropna(subset=cols)
    act_s = ss[(ss["activo"] == 1)].dropna(subset=cols)
    ent = act_r[~act_r["par_id"].isin(hold)][cols].values
    hol = act_r[act_r["par_id"].isin(hold)][cols].values
    sin = act_s[cols].values
    mu, sd = ent.mean(0), ent.std(0) + 1e-9
    nn = NearestNeighbors(n_neighbors=1).fit((ent - mu) / sd)
    m = min(20_000, len(hol), len(sin))
    d_hol = nn.kneighbors((hol[rng.choice(len(hol), m, replace=False)] - mu) / sd)[0].ravel()
    d_sin = nn.kneighbors((sin[rng.choice(len(sin), m, replace=False)] - mu) / sd)[0].ravel()
    p5 = np.percentile(d_hol, 5)
    add("Privacidad", "Prop. sintéticos con DCR < P5(real holdout)", float((d_sin < p5).mean()),
        cr["max_prop_dcr_bajo_p5"], nota=f"Referencia esperada ≈0,05; DCR mediana sint {np.median(d_sin):.3f} "
                                         f"vs holdout {np.median(d_hol):.3f}")
    add("Privacidad", "Prop. sintéticos con DCR = 0 (copia exacta de rasgos)", float((d_sin < 1e-9).mean()),
        nota="Informativa")
    # Trayectorias: ¿se reproduce la secuencia exacta de actividad del par real?
    m_r = real.pivot_table(index="par_id", columns="mes", values="activo")
    m_s = sint.pivot_table(index="par_id", columns="mes", values="activo").reindex_like(m_r)
    largos = m_r.notna().sum(1) >= 12
    iguales = ((m_r == m_s) | (m_r.isna() & m_s.isna())).all(1)
    add("Privacidad", "% pares (≥12 meses) con secuencia de actividad idéntica",
        float(100 * iguales[largos].mean()), nota="Informativa; esperado bajo")

    res = pd.DataFrame(filas)
    criterios = res.dropna(subset=["umbral"])
    resumen = {"criterios_evaluados": int(len(criterios)), "criterios_cumplidos": int(criterios["cumple"].sum()),
               "aprobado": bool(criterios["cumple"].all())}
    return res, resumen, {"mensual_real": mr, "mensual_sint": ms, "dcr_hol": d_hol, "dcr_sin": d_sin}


### Protocolo de generación (celda más lenta: unas 3 horas)
300 épocas; si no se cumplen todos los criterios, 600 y luego 1 200. Se conserva la versión con más criterios cumplidos y, en empate, la de menor error en los totales mensuales.

In [ ]:
def main(modo):
    fin = PERIODO_FIN          # una sola base sintética: todo el estudio se modela dentro de ella
    panel = pd.read_pickle(DIR_SEGURO / "panel_real.pkl")
    historial, mejor = [], None
    epochs = CTGAN_EPOCHS
    max_iter = SINTETICO_MAX_ITER_MEJORA
    for it in range(max_iter + 1):
        t0 = time.perf_counter()
        gen = GeneradorHibrido(epochs=epochs, semilla=SEMILLA + it).ajustar(panel, fin)
        t_fit = time.perf_counter() - t0
        sint = gen.generar()
        res, resumen, extra = validar(panel, sint)
        wape = float(res.loc[res["metrica"] == "WAPE ventas mensuales agregadas", "valor"].iat[0])
        registro = {"iteracion": it, "epochs": epochs, "seg_ajuste": round(t_fit, 1), **resumen,
                    "wape_agregado": wape}
        historial.append(registro)
        print(registro, flush=True)
        clave = (resumen["criterios_cumplidos"], -wape)
        if mejor is None or clave > mejor[0]:
            mejor = (clave, it, sint, res, extra, gen)
        if resumen["aprobado"]:
            break
        epochs *= 2

    _, it, sint, res, extra, gen = mejor
    sint.to_pickle(DIR_SEGURO / f"sintetico_{modo}.pkl")
    res.to_csv(DIR_TABLAS / f"C_validacion_sintetica_{modo}.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(historial).to_csv(DIR_TABLAS / f"C_protocolo_mejora_{modo}.csv", index=False,
                                   encoding="utf-8-sig")
    gen.ctgan.loss_values.to_csv(DIR_TABLAS / "B_ctgan_curva_perdida.csv", index=False, encoding="utf-8-sig")
    with open(DIR_SEGURO / f"extra_validacion_{modo}.pkl", "wb") as f:
        pickle.dump(extra, f)
    guardar_json({"modo": modo, "fin_ajuste": fin, "iteracion_elegida": it, "historial": historial,
                  "rho_residuos": gen.rho}, DIR_PUBLICABLE / f"C_resumen_sintetico_{modo}.json")
    # versión publicable: sin par_id interno, con identificadores re-tokenizados
    sint.drop(columns=["par_id"]).assign(mes=lambda d: d["mes"].astype(str),
                                          primer_mes=lambda d: d["primer_mes"].astype(str)) \
        .to_csv(DIR_PUBLICABLE / f"base_sintetica_{modo}.csv", index=False, encoding="utf-8-sig")


main("estudio")

## Paso 3. Tipos de relación en la base real y en la sintética

In [ ]:
# clasificar_intermitencia() está definida en el Paso 1; si saltaste ese paso, ejecuta antes su celda de funciones
r = pd.read_pickle(DIR_SEGURO / "panel_real.pkl")
s = pd.read_pickle(DIR_SEGURO / "sintetico_estudio.pkl")
cr = clasificar_intermitencia(r).groupby("par_id")["segmento_sb"].first().value_counts()
cs = clasificar_intermitencia(s).groupby("par_id")["segmento_sb"].first().value_counts()
out = pd.DataFrame({"real": cr, "sintetica": cs}).fillna(0).astype(int)
out.to_csv(DIR_TABLAS / "C_segmentos_real_vs_sintetica.csv", encoding="utf-8-sig")
out

## Paso 4. Figuras de fidelidad y privacidad
Solo agregados indexados (real 2023 = 100) y distribuciones.

In [ ]:
def fidelidad():
    estilo()
    real = pd.read_pickle(DIR_SEGURO / "panel_real.pkl")
    sint = pd.read_pickle(DIR_SEGURO / "sintetico_estudio.pkl")
    mr, ms = real.groupby("mes")["ventas_usd"].sum(), sint.groupby("mes")["ventas_usd"].sum()
    base = mr[mr.index.year == 2023].mean()
    fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
    x = mr.index.to_timestamp()
    ax[0].plot(x, 100 * mr / base, color=COLOR_REAL, lw=2, label="Real")
    ax[0].plot(x, 100 * ms.reindex(mr.index) / base, color=COLOR_SINT, lw=2, ls="--", label="Sintética")
    ax[0].set_ylabel("Índice de venta")
    ax[0].legend()
    ax[1].plot(x, 100 * real.groupby("mes")["activo"].mean(), color=COLOR_REAL, lw=2)
    ax[1].plot(x, 100 * sint.groupby("mes")["activo"].mean().reindex(mr.index), color=COLOR_SINT, lw=2, ls="--")
    ax[1].set_ylabel("% con movimiento")
    guardar_figura(fig, "S1_fidelidad_evolucion")
    ra, sa = real[real.activo == 1], sint[sint.activo == 1]
    fig, ax = plt.subplots(figsize=(7, 3.6))
    b = np.linspace(0, 6.2, 50)
    ax.hist(np.log10(ra.ventas_usd.abs()), bins=b, density=True, alpha=0.55, color=COLOR_REAL, label="Real")
    ax.hist(np.log10(sa.ventas_usd.abs()), bins=b, density=True, histtype="step", lw=2, color=COLOR_SINT, label="Sintética")
    ax.set_xlabel("log10(|venta neta mensual| en USD)")
    ax.set_ylabel("Densidad")
    ax.legend()
    guardar_figura(fig, "S2_fidelidad_distribucion")
    with open(DIR_SEGURO / "extra_validacion_estudio.pkl", "rb") as f:
        ex = pickle.load(f)
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    b = np.linspace(0, np.percentile(np.r_[ex["dcr_hol"], ex["dcr_sin"]], 99), 50)
    ax.hist(ex["dcr_hol"], bins=b, density=True, alpha=0.55, color=COLOR_REAL, label="Real reservado → real")
    ax.hist(ex["dcr_sin"], bins=b, density=True, histtype="step", lw=2, color=COLOR_SINT, label="Sintético → real")
    ax.axvline(np.percentile(ex["dcr_hol"], 5), color=GRISES[0], ls=":", lw=1)
    ax.set_xlabel("Distancia al registro real más cercano (DCR)")
    ax.set_ylabel("Densidad")
    ax.legend()
    guardar_figura(fig, "S3_privacidad_dcr")


fidelidad()

## Paso 5. Series externas y exportación de los insumos públicos

Lee las fuentes oficiales (INEC, BCE, EIA, NOAA, MSP) desde `Data/Datos_Macro/`, ubica cada serie en su mes de publicación y escribe los tres archivos de `Data/publico/` (categorías con menos de 5 clientes agrupadas en «Otros»).

In [ ]:
# DIR_MACRO y RUTA_COMPLETADOS se definen en el Paso 0

MESES_ES = {"ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6, "jul": 7, "ago": 8,
            "sep": 9, "oct": 10, "nov": 11, "dic": 12}

# variable: (fuente APA corta, frecuencia, rezago de publicación en meses)
CATALOGO = {
    "ipc_general":  ("INEC, s. f.", "mensual", 1),
    "ipc_salud":    ("INEC, s. f.", "mensual", 1),
    "desempleo":    ("INEC, 2026", "mensual", 1),
    "imae":         ("BCE, 2026", "mensual", 2),
    "riesgo_pais":  ("BCE, s. f.-a", "diaria→mensual", 0),
    "tasa_activa":  ("BCE, 2026", "mensual", 0),
    "remesas":      ("BCE, s. f.-b", "trimestral", 3),
    "dengue_casos": ("MSP, 2025", "semanal/casos→mensual", 1),
    "petroleo_wti": ("U.S. EIA, 2026", "mensual", 0),
    "oni":          ("NOAA CPC, 2026", "trimestral móvil→mes central", 2),
}


def _mes_es(txt):
    a, b = str(txt).strip().lower().split("-")
    return pd.Period(year=2000 + int(b), month=MESES_ES[a[:3]], freq="M")


def leer_series():
    s = {}
    # IPC nacional: índice general y división 06 Salud
    ipc = pd.read_excel(DIR_MACRO / "Tabulados_y_series_historicas_Excel/Series IPC/ipc_ind_nac_reg_ciud_12_2025.xlsx",
                        sheet_name="1. NACIONAL", header=None)
    fechas = [_mes_es(x) for x in ipc.iloc[4, 4:]]
    gen = ipc[ipc[3].astype(str).str.strip() == "GENERAL"].iloc[0, 4:]
    sal = ipc[(ipc[0] == "División") & (ipc[3].astype(str).str.strip() == "Salud")].iloc[0, 4:]
    s["ipc_general"] = pd.Series(gen.values.astype(float), index=fechas)
    s["ipc_salud"] = pd.Series(sal.values.astype(float), index=fechas)

    # ENEMDU: tasa de desempleo nacional
    t = pd.read_excel(DIR_MACRO / "202605_Tabulados_Mercado_Laboral_EXCEL.XLSX", sheet_name="2. Tasas", header=None)
    t = t[t[0].astype(str).str.startswith("ENEMDU") & (t[2].astype(str).str.strip() == "Desempleo (%)")]  # ENEMDU* = encuesta continua
    s["desempleo"] = pd.Series(t[3].astype(float).values, index=[_mes_es(x) for x in t[1]])

    # IMAEc (índice de volumen, serie ajustada; total)
    im = pd.read_excel(DIR_MACRO / "IMAEc_202607.xlsx", sheet_name="Of_Ivol_ajus_clas", header=None)
    im = im[im[0].astype(str).str.match(r"^\d{4}\.[a-z]{3}$")]
    idx = [pd.Period(year=int(x[:4]), month=MESES_ES[x[5:8]], freq="M") for x in im[0]]
    s["imae"] = pd.Series(im[8].astype(float).values, index=idx)

    # Riesgo país diario -> promedio mensual
    rp = pd.read_excel(DIR_MACRO / "Riesgo País.xlsx")
    rp.columns = ["fecha", "valor"]
    rp["fecha"] = pd.to_datetime(rp["fecha"])
    s["riesgo_pais"] = rp.groupby(rp["fecha"].dt.to_period("M"))["valor"].mean()

    # Tasa activa referencial
    ta = pd.read_excel(DIR_MACRO / "tasa_activa_referencial_BCE_2015_2026.xlsx", sheet_name="Serie mensual")
    s["tasa_activa"] = pd.Series(ta["tasa_activa_referencial"].values,
                                 index=pd.to_datetime(ta["fecha"]).dt.to_period("M"))

    # Remesas trimestrales (miles USD): el valor del trimestre se publica con rezago tras su cierre
    r = pd.read_excel(DIR_MACRO / "Serie_histórica_remesas_de_trabajadores.xlsx", sheet_name="MTR_t", header=None)
    r = r[r[1].astype(str).str.match(r"^\d{4}\.(I|II|III|IV)$")]
    rom = {"I": 3, "II": 6, "III": 9, "IV": 12}
    s["remesas"] = pd.Series(r[2].astype(float).values,
                             index=[pd.Period(year=int(x[:4]), month=rom[x[5:]], freq="M") for x in r[1]])

    # Dengue: casos notificados por semana epidemiológica (MSP, vigilancia SIVE-Alerta) -> mes
    # La semana se asigna al mes de su jueves; se descarta el último mes si está incompleto.
    dg = pd.read_excel(DIR_MACRO / "Datos_Dengue_MSP_Ene2021_Ago2025.xlsx", sheet_name="DSSA_2021_2024",
                       usecols=[0, 1, 32])
    dg.columns = ["anio", "semana", "total"]
    sem = dg.groupby(["anio", "semana"])["total"].sum().reset_index()
    jueves = [pd.Timestamp.fromisocalendar(int(a), min(int(w), 52), 4) for a, w in zip(sem["anio"], sem["semana"])]
    sem["mes"] = pd.DatetimeIndex(jueves).to_period("M")
    mensual = sem.groupby("mes")["total"].sum().astype(float)
    ultimo_jueves = max(jueves)
    if (ultimo_jueves + pd.Timedelta(days=7)).to_period("M") == mensual.index.max():
        mensual = mensual.iloc[:-1]
    s["dengue_casos"] = mensual

    # Meses completados desde fuentes oficiales (Data/exogenas_completadas.csv, con fuente por valor)
    comp = pd.read_csv(RUTA_COMPLETADOS)
    comp["mes"] = pd.PeriodIndex(comp["mes"], freq="M")
    for v, g in comp.groupby("variable"):
        ser = pd.Series(g["valor"].values, index=g["mes"])
        if v == "ipc_general_var_a":
            continue                      # se aplica tras calcular la variación (ver tabla_disponibilidad)
        s[v] = ser if v not in s else s[v].combine_first(ser)
    s["_ipc_var_a_2026"] = pd.Series(comp.loc[comp.variable == "ipc_general_var_a", "valor"].values,
                                     index=comp.loc[comp.variable == "ipc_general_var_a", "mes"])
    return s


def tabla_disponibilidad(series):
    """Valor que se conocería al cierre de cada mes origen t, y antigüedad del dato."""
    meses = pd.period_range("2022-01", PERIODO_FIN, freq="M")
    out, meta = pd.DataFrame(index=meses), []
    series = dict(series)
    ipc26 = series.pop("_ipc_var_a_2026", None)
    if ipc26 is not None:
        # variación anual: calculada con el índice base 2014 hasta 2025-12 y oficial INEC desde 2026-01
        base = series["ipc_general"].sort_index()
        var = (100 * base.pct_change(12)).dropna()
        series["ipc_general_var_a"] = var.combine_first(ipc26).sort_index()
        CATALOGO["ipc_general_var_a"] = ("INEC, s. f.; INEC, 2026", "mensual", 1)
    for v, ser in series.items():
        lag = CATALOGO[v][2]
        ser = ser.sort_index()
        publicado = pd.Series(ser.values, index=ser.index + lag)  # mes en que el dato ya es público
        alineado = publicado.reindex(meses.union(publicado.index)).ffill().reindex(meses)
        ult_pub = pd.Series(publicado.index, index=publicado.index).reindex(
            meses.union(publicado.index)).ffill().reindex(meses)
        edad = pd.Series([(m - u).n if pd.notna(u) else np.nan for m, u in zip(meses, ult_pub)], index=meses)
        if CATALOGO[v][1] == "trimestral":      # antigüedad en exceso sobre la cadencia normal (2 meses)
            edad = (edad - 2).clip(lower=0)
        out[v] = alineado
        meta.append({"variable": v, "fuente": CATALOGO[v][0], "frecuencia": CATALOGO[v][1],
                     "rezago_publicacion_meses": lag, "primer_dato": str(ser.index.min()),
                     "ultimo_dato": str(ser.index.max()),
                     "antiguedad_max_en_prueba": float(edad[edad.index > pd.Period(FIN_VALIDACION, "M")].max()),
                     "antiguedad_max_en_entrenamiento": float(
                         edad[(edad.index >= pd.Period(PERIODO_INICIO, "M")) &
                              (edad.index <= pd.Period(FIN_ENTRENAMIENTO, "M"))].max())})
    meta = pd.DataFrame(meta)
    # transformaciones estacionarias: variación interanual de índices y niveles
    if "ipc_general_var_a" not in out:
        out["ipc_general_var_a"] = 100 * out["ipc_general"].pct_change(12)
    out["ipc_salud_var_a"] = 100 * out["ipc_salud"].pct_change(12)
    out["imae_var_a"] = 100 * out["imae"].pct_change(12)
    out["remesas_var_a"] = 100 * out["remesas"].pct_change(12)
    out["dengue_log"] = np.log1p(out["dengue_casos"])
    return out, meta


def calendario(meses):
    """Variables conocidas de antemano para el mes objetivo."""
    import holidays
    fer = holidays.country_holidays("EC", years=range(2022, 2028))
    filas = []
    for m in meses:
        dias = pd.date_range(m.start_time, m.end_time, freq="D")
        feriados_hab = sum(1 for d in dias if d in fer and d.weekday() < 5)
        habiles = int(sum(d.weekday() < 5 for d in dias)) - feriados_hab
        filas.append({"mes": m, "feriados_habiles": feriados_hab, "dias_habiles": habiles,
                      # IVA general 15 % desde 1-abr-2024; medicamentos de la lista oficial: tarifa 0 %
                      "iva_15": int(m >= pd.Period("2024-04", "M"))})
    return pd.DataFrame(filas).set_index("mes")


VARIABLES_MACRO_MODELO = ["ipc_general_var_a", "ipc_salud_var_a", "desempleo", "imae_var_a",
                          "riesgo_pais", "tasa_activa", "remesas_var_a", "dengue_log", "petroleo_wti", "oni"]


def preparar_exogenas(max_antiguedad=3):
    series = leer_series()
    disp, meta = tabla_disponibilidad(series)
    # una variable se admite en el modelo principal solo si en la ventana de prueba su
    # último dato publicado no tiene más de `max_antiguedad` meses (si no, sería una
    # constante artificial justo donde se evalúa)
    base = {"ipc_salud_var_a": "ipc_salud", "imae_var_a": "imae",
            "remesas_var_a": "remesas", "dengue_log": "dengue_casos"}
    edad = meta.set_index("variable")["antiguedad_max_en_prueba"]
    meta["admitida_modelo"] = meta["variable"].map(lambda v: bool(edad[v] <= max_antiguedad))
    admitidas = [v for v in VARIABLES_MACRO_MODELO if edad[base.get(v, v)] <= max_antiguedad]
    meta.to_csv(DIR_TABLAS / "D_disponibilidad_exogenas.csv", index=False, encoding="utf-8-sig")
    disp.to_csv(DIR_TABLAS / "D_exogenas_alineadas_por_origen.csv", encoding="utf-8-sig")
    return disp, meta, admitidas


In [ ]:
K_MIN = 5
DIR_PUBLICO.mkdir(parents=True, exist_ok=True)
real = pd.read_pickle(DIR_SEGURO / "panel_real.pkl")
sint = pd.read_pickle(DIR_SEGURO / "sintetico_estudio.pkl")
atr = real.groupby("par_id").agg(cliente_real=("cliente", "first"), tipo_cliente=("tratamiento", "first"),
                                 detalle_negocio=("detalle_neg", "first"), provincia=("provincia", "first"),
                                 region=("region", "first"))
for c in ["tipo_cliente", "detalle_negocio", "provincia", "region"]:
    atr[c] = atr[c].astype(str).replace({"nan": "Sin dato"})
    n = atr.groupby(c)["cliente_real"].nunique()
    raras = n[n < K_MIN].index
    atr.loc[atr[c].isin(raras), c] = "Otros"
pub = sint.join(atr.drop(columns="cliente_real"), on="par_id")
pub = pub.rename(columns={"sku": "linea", "ventas_usd": "venta_neta"})
pub["mes"] = pub["mes"].astype(str)
cols = ["cliente", "linea", "mes", "venta_neta", "unidades", "tipo_cliente", "detalle_negocio", "provincia", "region"]
pub[cols].sort_values(["cliente", "linea", "mes"]).to_csv(DIR_PUBLICO / "base_sintetica.csv", index=False,
                                                          encoding="utf-8")

exog, meta, adm = preparar_exogenas()
exog.index = exog.index.astype(str)
exog.to_csv(DIR_PUBLICO / "exogenas_alineadas.csv", encoding="utf-8")
json.dump({"admitidas": adm, "nota": "Series alineadas al mes en que se publican (rezago de publicación)."},
          open(DIR_PUBLICO / "exogenas_admitidas.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(pub[cols].shape, {c: pub.drop_duplicates("cliente")[c].nunique() for c in cols[5:]})


### Listo
Los tres archivos de `Data/publico/` (`base_sintetica.csv`, `exogenas_alineadas.csv`, `exogenas_admitidas.json`) son los únicos insumos del análisis reproducible (`python reproducir.py`).